In [1]:
# -*- coding: utf-8 -*-
"""
Подготовка данных для предсказания банкротства
Запускать в ядре: Python (Bankruptcy Project)
"""

import sys
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("="*60)

# Проверяем версии библиотек
import numpy as np
import pandas as pd
import sklearn
import pyarrow
import fastparquet

print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"PyArrow: {pyarrow.__version__}")
print(f"FastParquet: {fastparquet.__version__}")

# Проверяем совместимость NumPy
if np.__version__.startswith('1.'):
    print("✅ NumPy 1.x - совместим с библиотеками")
else:
    print(f"⚠️ NumPy {np.__version__} - могут быть проблемы")
    print("Рекомендуется использовать NumPy 1.x")
    
print("="*60)
print("✅ Все библиотеки загружены корректно!")
print("="*60)

# --- Импорт остальных библиотек ---
import os
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

print("\n✅ Все необходимые библиотеки импортированы!")

ПРОВЕРКА ОКРУЖЕНИЯ
Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
NumPy: 1.26.4
Pandas: 3.0.5
Scikit-learn: 1.8.0
PyArrow: 23.0.0
FastParquet: 2026.5.0
✅ NumPy 1.x - совместим с библиотеками
✅ Все библиотеки загружены корректно!

✅ Все необходимые библиотеки импортированы!


In [18]:
# --- 1. Конфигурация ---
DATA_DIR = r"C:\Users\afina\Documents\Inforshare\Final project\data"
BASE_FILE = os.path.join(DATA_DIR, "company_years.parquet")
TASK_FILE = os.path.join(DATA_DIR, "company_years_h1.parquet")
SELECTED_HORIZON = "h1"

print(f"\n📁 Рабочая директория: {DATA_DIR}")
print(f"📊 Горизонт прогнозирования: {SELECTED_HORIZON}")


📁 Рабочая директория: C:\Users\afina\Documents\Inforshare\Final project\data
📊 Горизонт прогнозирования: h1


In [19]:
# --- 2. Загрузка данных ---
print("\n🔄 Загрузка признаков...")
try:
    df_features = pd.read_parquet(BASE_FILE)
    print(f"✅ Признаки загружены: {df_features.shape}")
except Exception as e:
    print(f"❌ Ошибка загрузки признаков: {e}")
    raise

print("\n🔄 Загрузка меток...")
try:
    df_labels = pd.read_parquet(TASK_FILE)
    print(f"✅ Метки загружены: {df_labels.shape}")
except Exception as e:
    print(f"❌ Ошибка загрузки меток: {e}")
    raise


🔄 Загрузка признаков...
✅ Признаки загружены: (1106879, 142)

🔄 Загрузка меток...
✅ Метки загружены: (1000087, 143)


In [20]:
# --- 3. Объединение данных ---
print("\n🔄 Объединение признаков и меток...")
df = df_features.join(df_labels[['main_label']])
print(f"✅ Объединенный датасет: {df.shape}")


🔄 Объединение признаков и меток...
✅ Объединенный датасет: (1106879, 143)


In [21]:
# --- 4. Анализ данных ---
print("\n📊 Анализ данных:")
print(f"Количество колонок: {len(df.columns)}")
print(f"Первые 10 колонок: {df.columns[:10].tolist()}")

# Баланс классов
class_counts = df['main_label'].value_counts()
print(f"\n📊 Баланс классов для {SELECTED_HORIZON}:")
print(f"Класс 0 (здоровые): {class_counts[0]:,} ({class_counts[0]/len(df)*100:.2f}%)")
print(f"Класс 1 (банкротство): {class_counts[1]:,} ({class_counts[1]/len(df)*100:.2f}%)")

# Пропуски
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0]
print(f"\n🔍 Колонки с пропусками: {len(null_cols)}")
if len(null_cols) > 0:
    print(f"Максимальное количество пропусков: {null_cols.max():,}")


📊 Анализ данных:
Количество колонок: 143
Первые 10 колонок: ['num', 'country', 'company', 'industry', 'link', 'emis_id', 'Current_assets/short_term_liabilities', 'Current_assets-inventories/short_term_liabilities', 'Current_assets-inventories-receivables/short_term_liabilities', 'Working_capital/total_assets']

📊 Баланс классов для h1:
Класс 0 (здоровые): 996,500 (90.03%)
Класс 1 (банкротство): 3,587 (0.32%)

🔍 Колонки с пропусками: 122
Максимальное количество пропусков: 1,106,879


In [22]:
# --- 5. ПРАВИЛЬНОЕ ОПРЕДЕЛЕНИЕ ТИПОВ КОЛОНОК ---
print("\n🔄 Анализ типов данных колонок...")

# Колонки, которые точно являются категориальными (из описания)
known_categorical = [
    'country', 'has_multiple_industries', 'naics_2digit', 'naics_3digit',
    'number_of_employees', 'operational_status', 'equity_ratio_classification',
    'Loss_flag', 'Insolvency_flag', 'legal_form', 'state', 'sector_1',
    'primary_naics_encoded', 'secondary_naics_encoded'
]

# Колонки, которые исключаем из обучения
cols_to_exclude = [
    'incorporation_date_1', 'incorporation_date_2'
]

# Определяем категориальные колонки на основе:
# 1. Известных категориальных
# 2. Типа данных object или category
# 3. Маленького количества уникальных значений
categorical_cols = []
for col in df.columns:
    if col == 'main_label':
        continue
    
    # Проверяем, есть ли в известных категориальных
    if col in known_categorical:
        categorical_cols.append(col)
        continue
    
    # Проверяем тип данных
    if df[col].dtype == 'object' or df[col].dtype == 'category':
        categorical_cols.append(col)
        continue
    
    # Проверяем количество уникальных значений
    n_unique = df[col].nunique()
    if n_unique < 20 and col not in ['year', 'main_label']:
        categorical_cols.append(col)
        continue

# Добавляем year как категориальную
if 'year' in df.columns and 'year' not in categorical_cols:
    categorical_cols.append('year')

# Фильтруем существующие колонки
categorical_cols = [col for col in categorical_cols if col in df.columns]
cols_to_exclude = [col for col in cols_to_exclude if col in df.columns]

# Все остальные колонки считаем числовыми
all_cols = df.columns.tolist()
all_cols.remove('main_label')
numeric_cols = [col for col in all_cols 
                if col not in categorical_cols 
                and col not in cols_to_exclude
                and col != 'main_label'
                and pd.api.types.is_numeric_dtype(df[col])]  # Важно: только числовые!

print(f"\n📋 Распределение колонок:")
print(f"Числовые: {len(numeric_cols)}")
print(f"Категориальные: {len(categorical_cols)}")
print(f"Исключенные: {len(cols_to_exclude)}")
print(f"Всего: {len(numeric_cols) + len(categorical_cols) + len(cols_to_exclude)}")

print(f"\n📋 Примеры категориальных колонок: {categorical_cols[:10]}")
print(f"📋 Примеры числовых колонок: {numeric_cols[:10]}")


🔄 Анализ типов данных колонок...

📋 Распределение колонок:
Числовые: 117
Категориальные: 22
Исключенные: 2
Всего: 141

📋 Примеры категориальных колонок: ['country', 'Equity_ratio_classification', 'state', 'number_of_employees', 'legal_form', 'operational_status', 'incorporation_date_1', 'incorporation_date_2', 'sector_1', 'sector_2']
📋 Примеры числовых колонок: ['num', 'emis_id', 'Current_assets/short_term_liabilities', 'Current_assets-inventories/short_term_liabilities', 'Current_assets-inventories-receivables/short_term_liabilities', 'Working_capital/total_assets', 'Working_capital/fixed_assets', 'Working_capital', 'Working_capital/total_operating_revenue', 'Working_capital/total_liabilities']


In [23]:
# --- 6. Проверка, что все числовые колонки действительно числовые ---
print("\n🔍 Проверка числовых колонок...")
non_numeric_in_numeric = []
for col in numeric_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        non_numeric_in_numeric.append(col)
        
if non_numeric_in_numeric:
    print(f"⚠️ Найдены нечисловые колонки в числовом списке: {non_numeric_in_numeric[:5]}")
    # Переносим их в категориальные
    for col in non_numeric_in_numeric:
        numeric_cols.remove(col)
        categorical_cols.append(col)
    print(f"✅ Исправлено. Перемещено в категориальные: {len(non_numeric_in_numeric)} колонок")


🔍 Проверка числовых колонок...


In [24]:
# --- 7. Разделение на X и y ---
print("\n🔄 Разделение на признаки (X) и целевую переменную (y)...")
y = df['main_label']
X = df.drop(columns=['main_label'] + cols_to_exclude)

# Убеждаемся, что все колонки в X имеют правильные типы
for col in categorical_cols:
    if col in X.columns:
        X[col] = X[col].astype(str)  # Приводим к строке для безопасного кодирования

print(f"Размер X: {X.shape}")
print(f"Размер y: {y.shape}")


🔄 Разделение на признаки (X) и целевую переменную (y)...
Размер X: (1106879, 140)
Размер y: (1106879,)


In [25]:
# --- 8. Создание препроцессора ---
print("\n🔄 Создание пайплайна предобработки...")

# Проверяем доступность колонок
available_categorical = [col for col in categorical_cols if col in X.columns]
available_numeric = [col for col in numeric_cols if col in X.columns]

print(f"Доступные категориальные: {len(available_categorical)}")
print(f"Доступные числовые: {len(available_numeric)}")

if len(available_numeric) == 0:
    print("⚠️ Предупреждение: Нет числовых колонок для обработки!")
    # Добавляем заглушку, если нет числовых колонок
    available_numeric = ['year'] if 'year' in X.columns else []

# Создаем трансформеры
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Создаем ColumnTransformer
transformers = []
if available_numeric:
    transformers.append(('num', numeric_transformer, available_numeric))
if available_categorical:
    transformers.append(('cat', categorical_transformer, available_categorical))

if not transformers:
    raise ValueError("Нет доступных колонок для трансформации!")

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder='drop'
)



🔄 Создание пайплайна предобработки...
Доступные категориальные: 20
Доступные числовые: 117


In [26]:
# --- 9. Применение предобработки ---
print("\n🔄 Применение предобработки к данным...")
print("⏳ Это может занять несколько минут...")

try:
    X_processed = preprocessor.fit_transform(X)
    print(f"✅ Данные обработаны! Размер: {X_processed.shape}")
except Exception as e:
    print(f"❌ Ошибка при предобработке: {e}")
    print("\n🔍 Диагностика:")
    print(f"Типы данных в X:")
    print(X.dtypes.value_counts())
    print(f"\nПример проблемной колонки:")
    for col in X.columns:
        if X[col].dtype == 'object':
            print(f"  {col}: {X[col].iloc[0] if len(X) > 0 else 'empty'}")
    raise


🔄 Применение предобработки к данным...
⏳ Это может занять несколько минут...
✅ Данные обработаны! Размер: (1106879, 135)


In [28]:
# --- 10. Создание DataFrame (исправленная версия) ---
print("\n🔄 Создание DataFrame из обработанных данных...")

# Получаем реальное количество колонок в X_processed
actual_num_cols = X_processed.shape[1]
print(f"Реальное количество колонок в X_processed: {actual_num_cols}")

# Получаем имена колонок из трансформеров
feature_names = available_numeric + available_categorical
print(f"Количество имен колонок: {len(feature_names)}")

# Если количество не совпадает, нужно определить реальные имена
if len(feature_names) != actual_num_cols:
    print(f"⚠️ Несоответствие: {len(feature_names)} имен vs {actual_num_cols} колонок")
    
    # Получаем имена колонок из ColumnTransformer
    # Для числовых колонок
    num_cols = []
    if 'num' in preprocessor.named_transformers_:
        # Получаем трансформер для числовых
        num_transformer = preprocessor.named_transformers_['num']
        # Имена числовых колонок
        num_cols = available_numeric
    
    # Для категориальных колонок
    cat_cols = []
    if 'cat' in preprocessor.named_transformers_:
        cat_transformer = preprocessor.named_transformers_['cat']
        # Имена категориальных колонок
        cat_cols = available_categorical
    
    # Объединяем в правильном порядке
    all_cols = num_cols + cat_cols
    print(f"Пересобранные имена: {len(all_cols)}")
    
    # Если все еще не совпадает, берем только первые N
    if len(all_cols) != actual_num_cols:
        print(f"⚠️ Все еще не совпадает. Берем первые {actual_num_cols} имен...")
        all_cols = all_cols[:actual_num_cols]
    
    feature_names = all_cols

# Создаем DataFrame
X_processed_df = pd.DataFrame(X_processed, columns=feature_names)
print(f"✅ Создан DataFrame с {X_processed_df.shape[1]} признаками")
print(f"✅ Имена колонок: {X_processed_df.columns[:5].tolist()}...")


🔄 Создание DataFrame из обработанных данных...
Реальное количество колонок в X_processed: 135
Количество имен колонок: 137
⚠️ Несоответствие: 137 имен vs 135 колонок
Пересобранные имена: 137
⚠️ Все еще не совпадает. Берем первые 135 имен...
✅ Создан DataFrame с 135 признаками
✅ Имена колонок: ['num', 'emis_id', 'Current_assets/short_term_liabilities', 'Current_assets-inventories/short_term_liabilities', 'Current_assets-inventories-receivables/short_term_liabilities']...


In [29]:
# --- 11. Создание ID компании для групповой кросс-валидации ---
print("\n🔄 Создание идентификаторов компаний...")

# Используем доступные статические признаки
available_id_cols = ['country', 'naics_2digit', 'naics_3digit']
available_id_cols = [col for col in available_id_cols if col in df.columns]

if available_id_cols:
    df['company_id'] = df.groupby(available_id_cols).ngroup()
else:
    # Если нет статических признаков, используем первые 3 категориальные
    cat_cols_for_id = available_categorical[:3] if available_categorical else ['year']
    df['company_id'] = df.groupby(cat_cols_for_id).ngroup()
    
print(f"Количество уникальных компаний: {df['company_id'].nunique()}")


🔄 Создание идентификаторов компаний...
Количество уникальных компаний: 241


In [30]:
# --- 12. Разбиение на обучающую и тестовую выборки ---
print("\n🔄 Разбиение на обучающую и тестовую выборки...")

try:
    gkf = GroupKFold(n_splits=5)
    train_idx, test_idx = next(gkf.split(X_processed_df, y, groups=df['company_id']))
    
    X_train, X_test = X_processed_df.iloc[train_idx], X_processed_df.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    print(f"✅ Разбиение выполнено:")
    print(f"Обучающая выборка: {X_train.shape}")
    print(f"Тестовая выборка: {X_test.shape}")
    
    # Баланс классов в выборках
    print(f"\n📊 Баланс классов в обучающей выборке:")
    train_counts = y_train.value_counts()
    print(f"Класс 0: {train_counts[0]:,} ({train_counts[0]/len(y_train)*100:.2f}%)")
    print(f"Класс 1: {train_counts[1]:,} ({train_counts[1]/len(y_train)*100:.2f}%)")
    
    print(f"\n📊 Баланс классов в тестовой выборке:")
    test_counts = y_test.value_counts()
    print(f"Класс 0: {test_counts[0]:,} ({test_counts[0]/len(y_test)*100:.2f}%)")
    print(f"Класс 1: {test_counts[1]:,} ({test_counts[1]/len(y_test)*100:.2f}%)")
    
except Exception as e:
    print(f"❌ Ошибка при разбиении: {e}")
    raise


🔄 Разбиение на обучающую и тестовую выборки...
✅ Разбиение выполнено:
Обучающая выборка: (885503, 135)
Тестовая выборка: (221376, 135)

📊 Баланс классов в обучающей выборке:
Класс 0: 787,700 (88.96%)
Класс 1: 2,862 (0.32%)

📊 Баланс классов в тестовой выборке:
Класс 0: 208,800 (94.32%)
Класс 1: 725 (0.33%)


In [33]:
# --- 13. Сохранение подготовленных данных ---
print("\n💾 Сохранение подготовленных данных...")

# Создаем папку для подготовленных данных
processed_dir = os.path.join(DATA_DIR, "processed")
os.makedirs(processed_dir, exist_ok=True)

# Сохраняем с правильным преобразованием типов
# X_train и X_test уже DataFrame, сохраняем как есть
X_train.to_parquet(os.path.join(processed_dir, f'X_train_{SELECTED_HORIZON}.parquet'), index=True)
X_test.to_parquet(os.path.join(processed_dir, f'X_test_{SELECTED_HORIZON}.parquet'), index=True)

# y_train и y_test - Series, преобразуем в DataFrame перед сохранением
y_train_df = pd.DataFrame(y_train, columns=['main_label'])
y_test_df = pd.DataFrame(y_test, columns=['main_label'])

y_train_df.to_parquet(os.path.join(processed_dir, f'y_train_{SELECTED_HORIZON}.parquet'), index=True)
y_test_df.to_parquet(os.path.join(processed_dir, f'y_test_{SELECTED_HORIZON}.parquet'), index=True)

# Сохраняем препроцессор
joblib.dump(preprocessor, os.path.join(processed_dir, f'preprocessor_{SELECTED_HORIZON}.pkl'))

print(f"✅ Данные сохранены в: {processed_dir}")
print(f"\nСохраненные файлы:")
print(f"  - X_train_{SELECTED_HORIZON}.parquet")
print(f"  - y_train_{SELECTED_HORIZON}.parquet")
print(f"  - X_test_{SELECTED_HORIZON}.parquet")
print(f"  - y_test_{SELECTED_HORIZON}.parquet")
print(f"  - preprocessor_{SELECTED_HORIZON}.pkl")


💾 Сохранение подготовленных данных...
✅ Данные сохранены в: C:\Users\afina\Documents\Inforshare\Final project\data\processed

Сохраненные файлы:
  - X_train_h1.parquet
  - y_train_h1.parquet
  - X_test_h1.parquet
  - y_test_h1.parquet
  - preprocessor_h1.pkl


In [34]:
# --- 14. Итоговая информация ---
print("\n" + "="*60)
print("📊 ИТОГОВАЯ ИНФОРМАЦИЯ")
print("="*60)
print(f"Количество признаков: {X_train.shape[1]}")
print(f"Количество обучающих примеров: {len(y_train):,}")
print(f"Количество тестовых примеров: {len(y_test):,}")
print(f"Доля банкротств в обучающей выборке: {y_train.mean()*100:.2f}%")
print(f"Доля банкротств в тестовой выборке: {y_test.mean()*100:.2f}%")
print("="*60)
print("\n🎉 ПОДГОТОВКА ДАННЫХ УСПЕШНО ЗАВЕРШЕНА!")
print("✅ Данные готовы для обучения модели машинного обучения!")


📊 ИТОГОВАЯ ИНФОРМАЦИЯ
Количество признаков: 135
Количество обучающих примеров: 885,503
Количество тестовых примеров: 221,376
Доля банкротств в обучающей выборке: 0.36%
Доля банкротств в тестовой выборке: 0.35%

🎉 ПОДГОТОВКА ДАННЫХ УСПЕШНО ЗАВЕРШЕНА!
✅ Данные готовы для обучения модели машинного обучения!
